# M06 lab solution: structured answers

**Brief.** Build a prompt template and parser that turn a question plus three passages into a JSON answer with a citations list.

The passages are the top three TF-IDF chunks of the Larkspur handbook for each question (`lab_kit.tfidf_retriever`), labelled `P1` to `P3`. A `PydanticOutputParser` defines `{"answer": str, "citations": [str]}` and supplies the format instructions. The solution runs 10 golden questions and counts parse failures and citations that point at a passage without an evidence span.

Key: `EXPERIENTIALLABS_API_KEY`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import pandas as pd
from langchain_core.exceptions import OutputParserException
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from helpers.utils import get_experientiallabs_llm

In [ ]:
from lab_kit import is_hit, load_handbook, load_jsonl, tfidf_retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for p in pages for c in splitter.split_text(p["text"])]
print(len(pages), "handbook files,", len(chunks), "chunks,", len(golden), "golden questions")

retrieve = tfidf_retriever(chunks, k=3)
questions = [g for g in golden if g["evidence_spans"]][:10]
print("questions:", ", ".join(g["id"] for g in questions))

## 1. The schema, the parser and the template

The template has a `{format_instructions}` slot filled once with `.partial()`. Printing the formatted prompt once confirms the schema reaches the model.

In [ ]:
class CitedAnswer(BaseModel):
    answer: str = Field(description="The answer to the question, using only the passages")
    citations: list[str] = Field(description="Ids of the passages the answer uses, such as P1")

parser = PydanticOutputParser(pydantic_object=CitedAnswer)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer customer questions about Larkspur Cloud using only the numbered passages. "
               "Cite the id of every passage you use. If the passages do not answer the question, say so "
               "and return an empty citations list.\n{format_instructions}"),
    ("human", "Passages:\n{passages}\n\nQuestion: {question}"),
]).partial(format_instructions=parser.get_format_instructions())

def format_passages(texts: list[str]) -> str:
    return "\n\n".join(f"[P{i}] {t}" for i, t in enumerate(texts, 1))

print(prompt.input_variables)
print(prompt.format(passages=format_passages(retrieve(questions[0]["question"])),
                    question=questions[0]["question"])[:1500])

## 2. Run 10 questions

The model call and the parse are separate steps, so a parse failure is counted instead of stopping the loop. A citation is *unsupported* when it names a passage with no evidence span, or an id that was not offered.

In [ ]:
llm = get_experientiallabs_llm()
chain = prompt | llm

rows = []
for g in questions:
    passages = retrieve(g["question"])
    ids = {f"P{i}": t for i, t in enumerate(passages, 1)}
    reply = chain.invoke({"passages": format_passages(passages), "question": g["question"]})
    try:
        parsed = parser.parse(reply.content)
    except OutputParserException:
        rows.append({"id": g["id"], "parsed": False, "answer": None, "citations": [], "unsupported": []})
        continue
    unsupported = [c for c in parsed.citations if c not in ids or not is_hit(ids[c], g["evidence_spans"])]
    rows.append({"id": g["id"], "parsed": True, "answer": parsed.answer, "citations": parsed.citations,
                 "unsupported": unsupported})

runs = pd.DataFrame(rows)
runs

## Results

In [ ]:
pd.DataFrame([{
    "questions": len(runs),
    "parse failures": int((~runs["parsed"]).sum()),
    "citations": int(runs["citations"].str.len().sum()),
    "citations to a passage without an evidence span": int(runs["unsupported"].str.len().sum()),
    "answers with no citation": int((runs["parsed"] & (runs["citations"].str.len() == 0)).sum()),
}])

## What this shows

- The parser turns a malformed reply into an `OutputParserException` the loop can count; a retry or a fallback can then handle it.
- The schema allows any string as a citation. An unsupported citation passes the parser, which is why the check against the retrieved ids and the evidence spans runs after it. Typing the ids as `Literal["P1", "P2", "P3"]` would move part of that check into the schema.
- A citation to a passage without an evidence span is not always wrong: a passage can support part of an answer in other words. Read those rows before calling them errors.